# Exp6: 基于集成学习的 Amazon 用户评论质量预测

## 一、案例简介

随着电商平台的兴起，以及疫情的持续影响，线上购物在我们的日常生活中扮演着越来越重要的角色。在进行线上商品挑选时，评论往往是我们十分关注的一个方面。然而目前电商网站的评论质量参差不齐，甚至有水军刷好评或者恶意差评的情况出现，严重影响了顾客的购物体验。因此，对于评论质量的预测成为电商平台越来越关注的话题，如果能自动对评论质量进行评估，就能根据预测结果避免展现低质量的评论。本案例中我们将基于集成学习的方法对 Amazon 现实场景中的评论质量进行预测。

## 二、作业说明

本案例中需要大家完成两种集成学习算法的实现（Bagging、AdaBoost.M1），其中基分类器要求使用 SVM 和决策树两种，因此，一共需要对比四组结果（[AUC](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics) 作为评价指标）：

* Bagging + SVM
* Bagging + 决策树
* AdaBoost.M1 + SVM
* AdaBoost.M1 + 决策树

注意集成学习的核心算法需要**手动进行实现**，基分类器可以调库。

### 基本要求
* 根据数据格式设计特征的表示
* 汇报不同组合下得到的 AUC
* 结合不同集成学习算法的特点分析结果之间的差异
* （使用 sklearn 等第三方库的集成学习算法会酌情扣分）

### 扩展要求
* 尝试其他基分类器（如 k-NN、朴素贝叶斯）
* 分析不同特征的影响
* 分析集成学习算法参数的影响

一些常用方法：
- 处理文本特征：sklearn.feature_extraction.text.TfidfVectorizer
- 大矩阵的处理：scipy.sparse
- SVM的运算速度较慢：用linearSVC代替SVC
- Ensemble的基类方法最好能输出probability而不是二分类结果，便于集成：CalibratedClassifierCV

------
------

## 1. 数据读取

本次数据来源于 Amazon 电商平台，包含超过 50,000 条用户在购买商品后留下的评论，各列的含义如下：

* reviewerID：用户 ID
* asin：商品 ID
* reviewText：英文评论文本
* overall：用户对商品的打分（1-5）
* votes_up：认为评论有用的点赞数（只在训练集出现）
* votes_all：该评论得到的总评价数（只在训练集出现）
* label：评论质量的 label，1 表示高质量，0 表示低质量（只在训练集出现）

评论质量的 label 来自于其他用户对评论的 votes，votes_up/votes_all ≥ 0.9 的作为高质量评论。此外测试集包含一个额外的列 ID，标识了每一个测试的样例。

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
train_df = pd.read_csv('./data/train.csv', sep='\t')
test_df = pd.read_csv('./data/test.csv', sep='\t')
len(train_df), len(test_df)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

## 2. 特征提取

In [ ]:
import scipy
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# tf/idf 处理文本特征
word_model = TfidfVectorizer()#stop_words='english')
train_X = word_model.fit_transform(train_df['reviewText'])
test_X = word_model.transform(test_df['reviewText']) 

# 拼上总评分特征 -- sparse 矩阵处理方式
train_X = scipy.sparse.hstack([train_X, train_df['overall'].values.reshape((-1, 1)) / 5])
test_X = scipy.sparse.hstack([test_X, test_df['overall'].values.reshape((-1, 1)) / 5])

## 3. Ensemble 算法实现

In [ ]:
import time
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.calibration import CalibratedClassifierCV

def construct_clf(clf_name):
    clf = None
    if clf_name == 'SVM':
        clf = svm.LinearSVC() # 用linearSVC代替SVC，可以加快速度
    elif clf_name == 'DTree' :
        clf = DecisionTreeClassifier(max_depth=10, class_weight='balanced')
    elif clf_name == 'NB' :
        clf = BernoulliNB()
    clf = CalibratedClassifierCV(clf, cv=2, method='sigmoid') # 概率校正，(1)使得本来无法输出probability的方法输出probability
                                                              # (2)使得输出的probability更准确
    return clf

In [ ]:
class Bagging(object):
    def __init__(self, clf, num_iter):
        self.clf = clf  # 分类器对象 -- 如果同时fit和predict，不需要存储，否则应当存下来
#         self.clf = []
        self.num_iter = num_iter  # Bagging 的分类器个数
        
    def fit_predict(self, X, Y, test_X, percentage=0.8):
        result = np.zeros(test_X.shape[0])  # 记录测试集的预测结果
        train_idx = np.arange(len(Y))
        for i in range(self.num_iter):
            sample_idx = np.random.choice(train_idx, size=int(len(Y)*percentage), replace=True)  # Bootstrap
            sample_train_X = X[sample_idx]
            sample_train_Y = Y[sample_idx]
            s = time.time()
            self.clf.fit(sample_train_X, sample_train_Y)
            e = time.time()
            print('Model {:<2d} training finish! (time: {:.1f} s)'.format(i,e-s))
            s = time.time()
            predict_proba = self.clf.predict_proba(test_X)[:, 1]
            e = time.time()
            print('Model {:<2d} prediction finish! (time: {:.1f} s)'.format(i,e-s))
            result += predict_proba  # 累加不同分类器的预测概率
        result /= self.num_iter  # 取平均（投票）
        return result

In [ ]:
class AdaBoostM1(object):
    def __init__(self, clf, num_iter):
        self.clf = clf  # 分类器对象
        self.num_iter = num_iter  # 迭代次数
        
    def fit_predict(self, X, Y, test_X):
        result_lst, beta_lst = list(), list()  # 记录每次迭代的预测结果和投票权重
        num_samples = len(Y)
        weight = np.ones(num_samples)  # 样本权重，【注意总和应为 num_samples！】
        for i in range(self.num_iter):
            s = time.time()
            self.clf.fit(X, Y, sample_weight=weight)  # 带权重的 fit
            e = time.time()
            print('Model {:<2d} training finish! (time: {:.1f} s)'.format(i,e-s))
            
            train_predict = self.clf.predict(X)  # 训练集预测结果
            error_flag = (train_predict != Y)  # 预测错误的位置
            error = weight[error_flag].sum() / num_samples  # 计算错误率
            print("error: {:3f}".format(error))
            if error > 0.5: # error过大，终止训练
                break
            beta = error / (1 - error)
            weight *= (1.0 - error_flag) * beta + error_flag  # 调整权重，正确位置乘上 beta，错误位置还是原来的
            weight = weight / weight.sum() * num_samples  # 归一化，让权重和等于 num_samples
            
            beta_lst.append(beta)
            s = time.time()
            predict_proba = self.clf.predict_proba(test_X)[:, 1]
            e = time.time()
            print('Model {:<2d} prediction finish! (time: {:.1f} s)'.format(i,e-s))
            result_lst.append(predict_proba)
            
        beta_lst = np.log(1 / np.array(beta_lst))
        beta_lst /= beta_lst.sum()  # 归一化投票权重
        
        print('\nVote Weight:\n', beta_lst)
        result = (np.array(result_lst) * beta_lst[:, None]).sum(0)  # 每一轮的预测结果加权求和
        return result#, result_lst

## 4. 测试并生成结果

In [ ]:
groundTruth = pd.read_csv("data/groundTruth.csv")
test_Y = groundTruth.Expected.to_numpy()

In [ ]:
# Bagging
np.random.seed(0)
clf = construct_clf('SVM')  # DTree, SVM, NB
runner = Bagging(clf, 10)
y_predict_bagging = runner.fit_predict(train_X.tocsr(), train_df['label'], test_X.tocsr())

In [ ]:
# AdaBoostM1
np.random.seed(0)
clf = construct_clf('DTree')  # DTree, SVM, NB
runner = AdaBoostM1(clf, 10)
y_predict_adaboost = runner.fit_predict(train_X.tocsr(), train_df['label'], test_X.tocsr())

In [ ]:
from sklearn.metrics import roc_auc_score
print("SVM+bagging auc=%.3f"%(roc_auc_score(test_Y, y_predict_bagging)))
print("DT+Adaboost auc=%.3f"%(roc_auc_score(test_Y, y_predict_adaboost)))

In [ ]:
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
model_bag = BaggingClassifier(svm.LinearSVC(), n_estimators=10)
model_bag = CalibratedClassifierCV(model_bag, cv=2, method='sigmoid') 
model_bag.fit(train_X.tocsr(), train_df['label'])
y_predict = model_bag.predict_proba(test_X.tocsr())[:,1]

## 5. 讨论

* 稀疏矩阵的使用
* 自行构建验证集进行线下测试，并汇报结果
* 文本建模方法 or 文本特征选择
* 引入文本长度、用户商品等特征

----

In [ ]:
import scipy
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# tf/idf 处理文本特征
word_model = TfidfVectorizer()#stop_words='english')
train_X = word_model.fit_transform(train_df['reviewText'])
test_X = word_model.transform(test_df['reviewText']) 

# 拼上总评分特征 -- sparse 矩阵处理方式
train_X_rate = scipy.sparse.hstack([train_X, train_df['overall'].values.reshape((-1, 1)) / 5])
test_X_rate = scipy.sparse.hstack([test_X, test_df['overall'].values.reshape((-1, 1)) / 5])

In [ ]:
groundTruth = pd.read_csv("groundTruth.csv")
test_Y = groundTruth.Expected.to_numpy()

In [ ]:
# 是否使用总评分特征，对基分类器的影响
import time
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

svm_clf = svm.LinearSVC()
svm_clf.fit(train_X,  train_df['label'])
pred = svm_clf.predict(test_X)
print("SVM without rating -- auc: %.3f"%(roc_auc_score(test_Y, pred)))
svm_clf2 = svm.LinearSVC()
svm_clf2.fit(train_X_rate,  train_df['label'])
pred = svm_clf2.predict(test_X_rate)
print("SVM with rating    -- auc: %.3f"%(roc_auc_score(test_Y, pred)))

svm_clf_cv = CalibratedClassifierCV(svm.LinearSVC(), cv=2, method='sigmoid')
svm_clf_cv.fit(train_X,  train_df['label'])
pred = svm_clf_cv.predict_proba(test_X)[:,1]
print("SVM (CV) without rating -- auc: %.3f"%(roc_auc_score(test_Y, pred)))
svm_clf_cv2 = CalibratedClassifierCV(svm.LinearSVC(), cv=2, method='sigmoid')
svm_clf_cv2.fit(train_X_rate,  train_df['label'])
pred = svm_clf_cv2.predict_proba(test_X_rate)[:,1]
print("SVM (CV) with rating    -- auc: %.3f"%(roc_auc_score(test_Y, pred)))

In [ ]:
dt_clf = DecisionTreeClassifier(max_depth=10)
dt_clf.fit(train_X,  train_df['label'])
pred = dt_clf.predict_proba(test_X)[:,1]
print("dt without rating -- auc: %.3f"%(roc_auc_score(test_Y, pred)))
dt_clf2 = DecisionTreeClassifier(max_depth=10)
dt_clf2.fit(train_X_rate,  train_df['label'])
pred = dt_clf2.predict_proba(test_X_rate)[:,1]
print("dt with rating    -- auc: %.3f"%(roc_auc_score(test_Y, pred)))

dt_clf_cv = CalibratedClassifierCV(DecisionTreeClassifier(max_depth=10), cv=2, method='sigmoid')
dt_clf_cv.fit(train_X,  train_df['label'])
pred = dt_clf_cv.predict_proba(test_X)[:,1]
print("dt (CV) without rating -- auc: %.3f"%(roc_auc_score(test_Y, pred)))
dt_clf_cv2 = CalibratedClassifierCV(DecisionTreeClassifier(max_depth=10), cv=2, method='sigmoid')
dt_clf_cv2.fit(train_X_rate,  train_df['label'])
pred = dt_clf_cv2.predict_proba(test_X_rate)[:,1]
print("dt (CV) with rating    -- auc: %.3f"%(roc_auc_score(test_Y, pred)))

In [ ]:
# 分析特征重要性
feat_importance = dt_clf2.tree_.compute_feature_importances(normalize=False)

In [ ]:
feat_importance[-1],sum((feat_importance[:-1])), feat_importance.shape

In [ ]:
from matplotlib import pyplot as plt
from sklearn import tree
fig = plt.figure(figsize=(25,20))
_ = tree.plot_tree(
    dt_clf2, 
    filled=True,
    class_names = ['F','T'],
    max_depth=2
)